**Question 1**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# ---------- 1) Load the dataset ----------
DATA_PATH = "stocks_df_combined_2026_09_18.parquet.brotli"
df_full = pd.read_parquet(DATA_PATH)

In [ ]:
# ---------- 2) Same truncated dataframe as the lecture (2000 onwards) ----------
TARGET = 'is_positive_growth_30d_future'

df_hw = df_full[df_full['Date'] >= '2000-01-01'].copy()
df_hw['Date'] = pd.to_datetime(df_hw['Date'])


In [ ]:
# ---------- 3) Add the week-of-month label, e.g. 'October_w1' ----------
df_hw['month_wom'] = (df_hw['Date'].dt.month_name() + '_w'
                      + ((df_hw['Date'].dt.day - 1) // 7 + 1).astype(str))


In [6]:
# ---------- 4) Categorical features, now including month_wom ----------
CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type', 'month_wom']

# Month and Weekday must be strings before get_dummies (same as lecture cell 27)
if pd.api.types.is_datetime64_any_dtype(df_hw['Month']):
    df_hw['Month'] = df_hw['Month'].dt.month.astype(str)
df_hw['Weekday'] = df_hw['Weekday'].astype(str)

In [7]:
# ---------- 5) Dummy variables ----------
dummy_variables = pd.get_dummies(df_hw[CATEGORICAL], dtype='int32')
DUMMIES = dummy_variables.columns.tolist()
MONTH_WOM_DUMMIES = [c for c in DUMMIES if c.startswith('month_wom_')]
print(f'Total dummies: {len(DUMMIES)}, month_wom dummies: {len(MONTH_WOM_DUMMIES)}')

# Keep the dummies in the dataframe (they are features for the next tasks)
df_hw = pd.concat([df_hw, dummy_variables], axis=1)

Total dummies: 115, month_wom dummies: 60


In [8]:
# ---------- 6) Correlation of every dummy with the target ----------
corr_all = df_hw[DUMMIES + [TARGET]].corr()[TARGET]


In [9]:
# ---------- 7) Keep only month_wom dummies, add abs_corr, sort descending ----------
corr_df = pd.DataFrame(corr_all)
corr_df = corr_df.loc[corr_df.index.isin(MONTH_WOM_DUMMIES)].copy()
corr_df['abs_corr'] = corr_df[TARGET].abs()
corr_df = corr_df.sort_values(by='abs_corr', ascending=False)


In [10]:
# ---------- 8) Report the answer ----------
print(corr_df.head(10))
print(f'Most correlated month_wom dummy: {corr_df.index[0]}')
print(f'Absolute correlation (3 dp): {corr_df["abs_corr"].iloc[0]:.3f}')

                       is_positive_growth_30d_future  abs_corr
month_wom_October_w4                        0.024584  0.024584
month_wom_November_w3                       0.023251  0.023251
month_wom_February_w1                      -0.020861  0.020861
month_wom_January_w2                       -0.019846  0.019846
month_wom_January_w5                       -0.019792  0.019792
month_wom_November_w2                       0.019103  0.019103
month_wom_January_w3                       -0.018537  0.018537
month_wom_February_w2                      -0.018340  0.018340
month_wom_January_w4                       -0.018265  0.018265
month_wom_October_w3                        0.017856  0.017856
Most correlated month_wom dummy: month_wom_October_w4
Absolute correlation (3 dp): 0.025
